In [1]:
using ForwardDiff
using Plots
using Pkg
using DifferentialEquations, Plots, PlutoUI; plotly()
using LinearAlgebra

#make a function that will return the function defined before by eric in matrix form(?)
# and a function thats going to return the jacobian of function returned above

#delta_t = (1/c)*(M_9/(M_* + sum_M_p)) * (G(M_* + sum_M_p)/P_9^2) * sin((t-t_0)(2pi/P_9))

#differentiating with respect to the mass of planet 9, P of planet 9, and t_0
#forgot again why we're differentiating with respect to t_0 lol

#I'm still confused on why we are solving for delta_t, how is it going to fold into our two-body solver?
#is it just to get all of the values we need to differentiat with respect to into one equation?
#I think that it might be the time step that we have to update our positions/velocities over as we integrate

#make new enviroment for projects

M_star = 10 #unittsssss
M_planets = 1000 #don't know what units I want everything in yet
M_system = M_star+M_planets
t = 10 #years? I think that this isn't something I will be differentiating with respect to so...
G = 6.674*10^-11

6.674e-11

In [2]:
#Mass of planet 9 = x[1]
#orbital period of planet 9 = x[2]
#t_0 (figured out what this is and am now confused again) = x[3]

function Equation(x::Vector)
    
    delta_t = (1/(3*10^8))*(x[1]/(M_system))*cbrt((6.674*10^-11)*(M_system)/x[2]^2)*sin((t - x[3])*(2*pi / x[2]))
    
    return delta_t
end

Equation (generic function with 1 method)

In [3]:
function Jacobian(M_9, P_9, t_0)

    delta_t = (1/(3*10^8))*(M_9/(M_system))*cbrt((6.674*10^-11)*(M_system)/P_9^2)*sin((t - t_0)*(2*pi / P_9))
    
    J(x) = ForwardDiff.jacobian(delta_t, x)
    
    return J
end
#something weird is happening with the Jacobian when I try and print it

Jacobian (generic function with 1 method)

In [4]:
#how do you feed values into the jacobian if we're looking at a range of possibilities?
# loop through a list or array of all the values?
show(Jacobian(1, 1, 1))

var"#J#Jacobian##0"{Float64}(-2.9608057830764805e-29)

In [5]:
myequation(x, y, z) = (1/(3*10^8))*(x/(M_system))*cbrt.((6.674*10^-11)*(M_system)/y.^2)*sin((t .- z)*(2*pi / y))

myequation (generic function with 1 method)

In [6]:
#is the larger supposed planet orbiting the solary system? So couldn't you reduce to a one
# body system instead?

In [7]:
# steps I think I should try and take
# 1. reduce to one-body problem (?)
# 2. define system and coords - how do you include the jacobian?
# 3. identify orbit shape - conic stuff the thing Eric sent us was talking about
# 4. calculation position, velocity, & trejectory - this should be a function
# 5. motion can be estimated via Keplerian dynamics - circular orbit first

# would we utilize leap frog or Euler intergration?

# how do I calculate the initial position vectors?
# I'm unsure how to calculate the intial position vector for planet 9 since P is part of the 
# jacobian and is something were expecting to change

# need to calculate relative motion of ___

# could also start by 
# 1. calculate separation distance (r = r1 - r2)
# 2. calculate gravitational force on the two bodies via newtons law to find acceleration
# 3. update positions and velocities via delta_t (ex. v_new = v_old + a(delta_t))
# idk what values this is getting me

# what values do I actually need? - I need all the values that will allow me to plot the orbit of the system
# Should I make it possible to visualize the actual orbits I would be modeling? - yes

# I think that M_9, P_9, and t_0 will end up being lists/arrays I'll have to iterate through
# - do I need to do anything to my functions to make sure that'll work? (i don't think so but I'll look into it)

In [8]:
#velocity and position vectors?


In [9]:
#calculates position vector
function separationDistance(r_system::Vector, r_9::Vector)
    
    r = r_9 - r_system
    
    return r
end

separationDistance (generic function with 1 method)

In [10]:
#finds acceleration for both bodies in the system
function acceleration(r::Vector, M_9)
    
    a_9 = (G*M_system/(norm(r)^3)) .* r
    a_system = (G*M_9/(norm(r)^3)) .* r
    
    return a_9, a_system
end

acceleration (generic function with 1 method)

In [11]:
# updates position and velocity with time-step delta_t (I don't have a position vector yet so...) 
#(the thing I'm looking at makes me feel like it's diff. from r here)
# acceleration of which body tho? (a = acceleration)
function stateUpdate(v::Vector, r::Vector, delta_t, a)

    v_new = v .+ a .* delta_t
    r_new = r .+ v_new .* delta_t

    return v_new, r_new

end

stateUpdate (generic function with 1 method)

In [12]:
#center of mass of the two-body system

function CenterOfMass(P_system, P_9, M_9)

    P_c = ((M_system*P_system)+(M_9*P_9))/(M_9+M_system) #check that planet 9 is m1 and P1

    return P_c
end

CenterOfMass (generic function with 1 method)

In [13]:
#should return two position vectors
function OrbitalRadii(r::Vector, M_9)

    #check that planet 9 is m1
    r_system = (M_9/(M_system+M_9)) .* r
    r_9 = (M_system/(M_system+M_9)) .* r

    return r_system, r_9
end

OrbitalRadii (generic function with 1 method)

In [14]:
#orbital period for a one-body system :/ - I don't think I can get it to two

function OrbitalPeriod(r::Vector, M_9)
    
    T = sqrt((4*pi^2 .* r^3)/((6.674*10^-11) .* (M_system + M_9)))
    
    return T
end

OrbitalPeriod (generic function with 1 method)

In [15]:
#v = relative velocity vector
#r = relative position vector - calculated via center of mass frame
#h = angular momentum vector
#I only need this when it's not an assumed circular orbit

function Eccentricity(v::Vector, r1::Vector, r2::Vector, h::Vector, r_constant)

    r = r1 - r2 #relative position of planet 9 to trappist system 
    
    e = 1/(G*(M_star + M_planets))*cross(v,h) - (r./r_constant) #I may not need the period I put in here

    return e

end

Eccentricity (generic function with 1 method)